__Instead of excluding the QSO targets, here only include QSO targets.__

In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/everest/sv1_perexp_lrg.fits'))
cat['EFFTIME_ELG'] = 8.60 * cat['TSNR2_ELG']
cat['EFFTIME_LRG'] = 12.15 * cat['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat['FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Remove "no data" fibers
mask = cat['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Apply LRG mask
mask = cat['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[mask]

# Remove QSO targets
mask = cat['SV1_DESI_TARGET'] & 2**2 ==0
print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat = cat[~mask]

# # Require a minimum depth
# min_depth = 500.
# mask = cat['EFFTIME_LRG']>min_depth
# print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
# cat = cat[mask]

# Julien's bad fibers list + my list of worst-performing fibers; notebooks/main/failures_analysis-everest.ipynb
bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211013.txt', dtype=int)
print(len(bad_fibers))
mask_bad = np.in1d(cat['FIBER'], bad_fibers)
print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
cat = cat[~mask_bad]
print(len(cat))

FIBERSTATUS 414186 91655 0.18119329987090804
No data 414186 0 0.0
LRG mask 371619 42567 0.10277266735234894
Remove QSO targets 360700 10919 0.029382243642009694
333
Bad fibers 9962 957 0.08764538877186556
9962


In [4]:
cat_1x = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/everest/sv1_1x_depth_lrg.fits'))
cat_1x['EFFTIME_ELG'] = 8.60 * cat_1x['TSNR2_ELG']
cat_1x['EFFTIME_LRG'] = 12.15 * cat_1x['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = cat_1x['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat_1x = cat_1x[mask]

# Remove "no data" fibers
mask = cat_1x['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat_1x = cat_1x[mask]

# Apply LRG mask
mask = cat_1x['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat_1x = cat_1x[mask]

# Remove QSO targets
mask = cat_1x['SV1_DESI_TARGET'] & 2**2 ==0
print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
cat_1x = cat_1x[~mask]

# # Require a minimum depth
# min_depth = 500.
# mask = cat_1x['EFFTIME_LRG']>min_depth
# print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
# cat_1x = cat_1x[mask]

# Julien's bad fibers list + my list of worst-performing fibers; notebooks/main/failures_analysis-everest.ipynb
bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211013.txt', dtype=int)
print(len(bad_fibers))
mask_bad = np.in1d(cat_1x['FIBER'], bad_fibers)
print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
cat_1x = cat_1x[~mask_bad]
print(len(cat_1x))

FIBERSTATUS 27179 4874 0.15206064954918416
No data 27179 0 0.0
LRG mask 25072 2107 0.07752308767798669
Remove QSO targets 24557 515 0.020540842373962986
333
Bad fibers 484 31 0.06019417475728155
484


In [5]:
columns = list(np.intersect1d(cat.colnames, cat_1x.colnames))
cat = cat[columns]
cat_1x = cat_1x[columns]

cat = vstack([cat, cat_1x])

In [6]:
deep = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/everest/sv1_cumulative_lrg.fits'))
deep['EFFTIME_ELG'] = 8.60 * deep['TSNR2_ELG']
deep['EFFTIME_LRG'] = 12.15 * deep['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = deep['COADD_FIBERSTATUS']==0
print('COADD_FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
deep = deep[mask]

mask = deep['ZWARN']==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
deep = deep[mask]

# Apply LRG mask
mask = deep['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
deep = deep[mask]

# Remove QSO targets
mask = deep['SV1_DESI_TARGET'] & 2**2 ==0
print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
deep = deep[~mask]

# Require a minimum depth
min_depth = 3000.
mask = deep['EFFTIME_LRG']>min_depth
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
deep = deep[mask]

# Julien's bad fibers list + my list of worst-performing fibers; notebooks/main/failures_analysis-everest.ipynb
bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211013.txt', dtype=int)
print(len(bad_fibers))
mask_bad = np.in1d(deep['FIBER'], bad_fibers)
print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
deep = deep[~mask_bad]
print(len(deep), len(np.unique(deep['TARGETID'])))

# Remove duplidates keeping the higher EFFTIME objects
deep.sort('EFFTIME_LRG', reverse=True)
_, idx_keep = np.unique(deep['TARGETID'], return_index=True)
deep = deep[idx_keep]
print(len(deep), len(np.unique(deep['TARGETID'])))

deep_columns_old = ['TARGETID', 'Z', 'ZERR', 'ZWARN', 'SPECTYPE', 'DELTACHI2', 'EFFTIME_LRG', 'EFFTIME_ELG']
deep_columns_new = ['TARGETID', 'Z_deep', 'ZERR_deep', 'ZWARN_deep', 'SPECTYPE_deep', 'DELTACHI2_deep', 'EFFTIME_LRG_deep', 'EFFTIME_ELG_deep']
deep.rename_columns(deep_columns_old, deep_columns_new)

cat = join(cat, deep[deep_columns_new], keys='TARGETID')

COADD_FIBERSTATUS 46403 8832 0.15989861500859961
No data 45392 1011 0.021787384436351098
LRG mask 41301 4091 0.09012601339443074
Remove QSO targets 39607 1694 0.041015956030120336
Min depth 1244 450 0.7343565525383707
333
Bad fibers 1147 97 0.07797427652733119
1147 1145
1145 1145


In [7]:
main = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/everest/main_cumulative_lrg.fits'))
main['EFFTIME_ELG'] = 8.60 * main['TSNR2_ELG']
main['EFFTIME_LRG'] = 12.15 * main['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = main['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
main = main[mask]

# Remove "no data" fibers
mask = main['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
main = main[mask]

# Apply LRG mask
mask = main['lrg_mask']==0
print('LRG mask', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
main = main[mask]

# Remove QSO targets
mask = main['DESI_TARGET'] & 2**2 ==0
print('Remove QSO targets', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
main = main[~mask]

# # Require a minimum depth
# min_depth = 500.
# mask = main['EFFTIME_LRG']>min_depth
# print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
# main = main[mask]

# Julien's bad fibers list + my list of worst-performing fibers; notebooks/main/failures_analysis-everest.ipynb
bad_fibers = np.loadtxt('/Users/rongpu/Documents/Data/desi_data/everest/misc/bad_fibers_20211013.txt', dtype=int)
print(len(bad_fibers))
mask_bad = np.in1d(main['FIBER'], bad_fibers)
print('Bad fibers', np.sum(~mask_bad), np.sum(mask_bad), np.sum(mask_bad)/len(mask_bad))
main = main[~mask_bad]
print(len(main))

FIBERSTATUS 340944 5119 0.014792104327824702
No data 340944 0 0.0
LRG mask 306665 34279 0.10054143789009339
Remove QSO targets 301954 4711 0.015362040011087017
333
Bad fibers 4367 344 0.07302059010825727
4367


In [8]:
mask = cat['EFFTIME_LRG']>800
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
cat = cat[mask]
print(len(cat))

Min depth 1894 6992 0.2131442718883637
1894


In [9]:
mask = main['EFFTIME_LRG']>800
print('Min depth', np.sum(mask), np.sum(~mask), np.sum(mask)/len(mask))
main = main[mask]
print(len(main))

Min depth 4261 106 0.9757270437371193
4261


In [10]:
cat['zfibermag'] = 22.5 - 2.5*np.log10(cat['FIBERFLUX_Z']) - 1.211 * cat['EBV']
main['zfibermag'] = 22.5 - 2.5*np.log10(main['FIBERFLUX_Z']) - 1.211 * main['EBV']

---------

In [11]:
# Catastrophic redshift failures
zdiff_threshold = 0.0033
mask_fail = np.abs((cat['Z'] - cat['Z_deep'])/(1 + cat['Z_deep'])) > zdiff_threshold
# Also reject objects with ZWARN!=0 or z>1.4 in the deep coadds
mask_deep_fail = (cat['ZWARN_deep']!=0) | (cat['Z_deep']>1.4)
# mask_deep_fail = (cat['ZWARN_deep']!=0) | (cat['Z_deep']>1.4) | (cat['DELTACHI2_deep']<25)
mask_fail |= mask_deep_fail
print('All SV LRGs:')
print('Catastrophic failure rate: {:.2f}% ({}/{})'.format(100*np.sum(mask_fail)/len(mask_fail), np.sum(mask_fail), len(mask_fail)))
print()

mask = cat['main_lrg'].copy()
print('Main LRGs in SV:')
print('Catastrophic failure rate: {:.2f}% ({}/{})'.format(100*np.sum(mask_fail & mask)/np.sum(mask), np.sum(mask_fail & mask), np.sum(mask)))

All SV LRGs:
Catastrophic failure rate: 13.83% (262/1894)

Main LRGs in SV:
Catastrophic failure rate: 12.46% (73/586)


In [12]:
# DELTACHI2 cut
q0 = cat['ZWARN']==0
q0 &= cat['Z']<1.4
q0 &= cat['DELTACHI2']>12.5

# Custom DELTACHI2 vs redshift cut
d = (10**(3 - 3.5*cat['Z']))
mask_remove = (d>30) & (cat['DELTACHI2']<30)
mask_remove |= (d<30) & (cat['DELTACHI2']<d)
mask_remove |= (cat['DELTACHI2']<10)
q = cat['ZWARN']==0
q &= cat['Z']<1.4
q &= (~mask_remove)

# Custom DELTACHI2 vs zfiber cut
q1 = cat['DELTACHI2'] > 10**(-(cat['zfibermag']-20)*0.4*2 + 2.17)
q1 |= cat['DELTACHI2'] > 10**(-(cat['zfibermag']-20)*0.4 + 2.1)
q1 &= cat['ZWARN']==0
q1 &= cat['Z']<1.4

In [13]:
print('########### DELTACHI2 cut ###########')
print('SV LRGs:')
print('Removed fraction: {:.2f}%'.format(100*np.sum(~q0)/len(q0)))
print('Catastrophic failure rate of removed objects: {:.2f}% ({}/{})'.format(100*np.sum((~q0) & mask_fail)/np.sum(~q0), np.sum((~q0) & mask_fail), np.sum(~q0)))
print('Catastrophic failure rate of remaining objects: {:.2f}% ({}/{})'.format(100*np.sum((q0) & mask_fail)/np.sum(q0), np.sum(q0 & mask_fail), np.sum(q0)))
print()

mask = cat['main_lrg'].copy()
print('Main LRGs in SV:')
print('Removed fraction: {:.2f}%'.format(100*np.sum(mask & (~q0))/np.sum(mask)))
print('Catastrophic failure rate of removed objects: {:.2f}% ({}/{})'.format(100*np.sum(mask & (~q0) & mask_fail)/np.sum(mask & (~q0)), np.sum(mask & (~q0) & mask_fail), np.sum(mask & (~q0))))
print('Catastrophic failure rate of remaining objects: {:.2f}% ({}/{})'.format(100*np.sum(mask & q0 & mask_fail)/np.sum(mask & q0), np.sum(mask & q0 & mask_fail), np.sum(mask & q0)))

########### DELTACHI2 cut ###########
SV LRGs:
Removed fraction: 12.94%
Catastrophic failure rate of removed objects: 91.02% (223/245)
Catastrophic failure rate of remaining objects: 2.37% (39/1649)

Main LRGs in SV:
Removed fraction: 10.41%
Catastrophic failure rate of removed objects: 91.80% (56/61)
Catastrophic failure rate of remaining objects: 3.24% (17/525)


In [14]:
print('########### DELTACHI2 vs redshift cut ###########')
print('SV LRGs:')
print('Removed fraction: {:.2f}%'.format(100*np.sum(~q)/len(q)))
print('Catastrophic failure rate of removed objects: {:.2f}% ({}/{})'.format(100*np.sum((~q) & mask_fail)/np.sum(~q), np.sum((~q) & mask_fail), np.sum(~q)))
print('Catastrophic failure rate of remaining objects: {:.2f}% ({}/{})'.format(100*np.sum((q) & mask_fail)/np.sum(q), np.sum(q & mask_fail), np.sum(q)))
print()

mask = cat['main_lrg'].copy()
print('Main LRGs in SV:')
print('Removed fraction: {:.2f}%'.format(100*np.sum(mask & (~q))/np.sum(mask)))
print('Catastrophic failure rate of removed objects: {:.2f}% ({}/{})'.format(100*np.sum(mask & (~q) & mask_fail)/np.sum(mask & (~q)), np.sum(mask & (~q) & mask_fail), np.sum(mask & (~q))))
print('Catastrophic failure rate of remaining objects: {:.2f}% ({}/{})'.format(100*np.sum(mask & q & mask_fail)/np.sum(mask & q), np.sum(mask & q & mask_fail), np.sum(mask & q)))

########### DELTACHI2 vs redshift cut ###########
SV LRGs:
Removed fraction: 14.10%
Catastrophic failure rate of removed objects: 86.89% (232/267)
Catastrophic failure rate of remaining objects: 1.84% (30/1627)

Main LRGs in SV:
Removed fraction: 12.29%
Catastrophic failure rate of removed objects: 84.72% (61/72)
Catastrophic failure rate of remaining objects: 2.33% (12/514)


In [15]:
print('########### DELTACHI2 vs zfiber cut ###########')
print('SV LRGs:')
print('Removed fraction: {:.2f}%'.format(100*np.sum(~q1)/len(q1)))
print('Catastrophic failure rate of removed objects: {:.2f}% ({}/{})'.format(100*np.sum((~q1) & mask_fail)/np.sum(~q1), np.sum((~q1) & mask_fail), np.sum(~q1)))
print('Catastrophic failure rate of remaining objects: {:.2f}% ({}/{})'.format(100*np.sum((q1) & mask_fail)/np.sum(q1), np.sum(q1 & mask_fail), np.sum(q1)))
print()

mask = cat['main_lrg'].copy()
print('Main LRGs in SV:')
print('Removed fraction: {:.2f}%'.format(100*np.sum(mask & (~q1))/np.sum(mask)))
print('Catastrophic failure rate of removed objects: {:.2f}% ({}/{})'.format(100*np.sum(mask & (~q1) & mask_fail)/np.sum(mask & (~q1)), np.sum(mask & (~q1) & mask_fail), np.sum(mask & (~q1))))
print('Catastrophic failure rate of remaining objects: {:.2f}% ({}/{})'.format(100*np.sum(mask & q1 & mask_fail)/np.sum(mask & q1), np.sum(mask & q1 & mask_fail), np.sum(mask & q1)))

########### DELTACHI2 vs zfiber cut ###########
SV LRGs:
Removed fraction: 16.16%
Catastrophic failure rate of removed objects: 78.76% (241/306)
Catastrophic failure rate of remaining objects: 1.32% (21/1588)

Main LRGs in SV:
Removed fraction: 17.92%
Catastrophic failure rate of removed objects: 62.86% (66/105)
Catastrophic failure rate of remaining objects: 1.46% (7/481)


In [16]:
# DELTACHI2 cut
mq0 = main['ZWARN']==0
mq0 &= main['Z']<1.4
mq0 &= main['DELTACHI2']>12.5

# Custom DELTACHI2 vs redshift cut
d = (10**(3 - 3.5*main['Z']))
mask_remove = (d>30) & (main['DELTACHI2']<30)
mask_remove |= (d<30) & (main['DELTACHI2']<d)
mask_remove |= (main['DELTACHI2']<10)
mq = main['ZWARN']==0
mq &= main['Z']<1.4
mq &= (~mask_remove)

# Custom DELTACHI2 vs zfiber cut
mq1 = main['DELTACHI2'] > 10**(-(main['zfibermag']-20)*0.4*2 + 2.17)
mq1 |= main['DELTACHI2'] > 10**(-(main['zfibermag']-20)*0.4 + 2.1)
mq1 &= main['ZWARN']==0
mq1 &= main['Z']<1.4

In [17]:
print('DELTACHI2 cut:')
print('Removed fraction: {:.3f}% ({}/{})'.format(100*np.sum(~mq0)/len(mq0), np.sum(~mq0), len(mq0)))
print()

print('DELTACHI2 vs redshift cut:')
print('Removed fraction: {:.3f}% ({}/{})'.format(100*np.sum(~mq)/len(mq), np.sum(~mq), len(mq)))
print()

print('DELTACHI2 vs zfiber cut:')
print('Removed fraction: {:.3f}% ({}/{})'.format(100*np.sum(~mq1)/len(mq1), np.sum(~mq1), len(mq1)))
print()

DELTACHI2 cut:
Removed fraction: 12.767% (544/4261)

DELTACHI2 vs redshift cut:
Removed fraction: 13.753% (586/4261)

DELTACHI2 vs zfiber cut:
Removed fraction: 19.831% (845/4261)

